# General relativity: orbits and infall around a black hole

*astro · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/black_hole_geodesics)

Geodesics in the Schwarzschild metric, with the Christoffel symbols obtained from $g_{\mu\nu}(x)$ by automatic differentiation and contracted with einsum. A bound orbit precesses instead of closing, the effective potential shows the innermost stable circular orbit at $6GM/c^2$ and the photon sphere at $3GM/c^2$, and a radial plunge reaches the horizon after a finite proper time while the far-away clock freezes.

**Vector form**

$$ds^2 = -\Big(1 - \frac{r_s}{r}\Big)c^2dt^2 + \Big(1 - \frac{r_s}{r}\Big)^{-1}dr^2 + r^2 d\Omega^2,\qquad r_s = \frac{2GM}{c^2}$$

**Index (tensor) form**

$$\frac{d^2x^\mu}{d\tau^2} = -\Gamma^\mu{}_{\alpha\beta}\,\dot x^\alpha\dot x^\beta,\qquad \Gamma^\mu{}_{\alpha\beta} = \tfrac12\,g^{\mu\sigma}\left(\partial_\alpha g_{\sigma\beta} + \partial_\beta g_{\sigma\alpha} - \partial_\sigma g_{\alpha\beta}\right)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x^\mu$ | `y[:, 0, :]` | (B, 4) | event $(t, r, \theta, \varphi)$ on each geodesic, $G = c = M = 1$ |
| $\dot x^\mu$ | `y[:, 1, :]` | (B, 4) | 4-velocity $dx^\mu/d\tau$ |
| $g_{\mu\nu}$ | `metric(x, M)` | (4, 4) | Schwarzschild metric, signature $(-, +, +, +)$ |
| $g^{\mu\sigma}$ | `torch.linalg.inv(g)` | (B, 4, 4) | inverse metric |
| $\partial_\alpha g_{\sigma\beta}$ | `vmap(jacrev(metric))(x)` | (B, 4, 4, 4) | metric derivatives by autograd, derivative index last |
| $\Gamma^\mu{}_{\alpha\beta}$ | `0.5 * einsum('zms,zsab->zmab', g_inv, d1 + d2 - d3)` | (B, 4, 4, 4) | repeated $\sigma$ ⇒ summed ($z$ = batch) |
| $\Gamma^\mu{}_{\alpha\beta}\dot x^\alpha\dot x^\beta$ | `einsum('zmab,za,zb->zm', Gamma, u, u)` | (B, 4) | repeated $\alpha, \beta$ ⇒ summed |
| $g_{\mu\nu}\dot x^\mu\dot x^\nu$ | `norm(y, M)` | (T, B) | $= -1$ for a massive particle, checks the integration |

Units: $G = c = M = 1$, so lengths are in $GM/c^2$ (1.5 km per solar mass) and the horizon is at $r_s = 2$. The orbit starts at periapsis with the exact Schwarzschild energy $E$ and angular momentum $L$ of an orbit with semi-latus rectum $p$ and eccentricity $e$. It is integrated in proper time $\tau$ with RK4; the grey Newtonian ellipse with the same $p$ and $e$ closes on itself. The perihelion advance approaches $6\pi GM/(c^2 p)$ for large $p$ (43″ per century for Mercury) and grows much larger close to the hole. Bound orbits exist only for $p > 6 + 2e$; below the ISCO ($r = 6$) circular orbits are unstable, and at $r = 3$ even light can circle. In the plunge (lower right), the falling clock reaches the horizon after $\tau = \tfrac{\pi}{2}\sqrt{r_0^3/2}$-ish (finite), but in Schwarzschild time $t$ the fall slows and freezes at $r = 2$; the run stops at $r = 2.1$, where these coordinates start to fail.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/black_hole_geodesics/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    # Many tiny autograd calls in float64: the CPU is the right device here.
    M, p, e = tensors.M, params["p"], params["e"]
    E, L = tensors.orbit_constants(p, e)
    a = p / (1 - e**2)
    tau_max = params["n_orbits"] * 2 * math.pi * a**1.5  # roughly n Newtonian periods

    tau, gr = solve.geodesic(tensors.bound_orbit(p, e), M, tau_max, params["dt"])
    newton = solve.newton(tensors.newton_orbit(p, e), M, 2 * math.pi * a**1.5, params["dt"] / 4)
    peri = solve.periapsis_angles(gr)
    precession = float((peri.diff() - 2 * math.pi).mean()) if len(peri) > 1 else float("nan")
    rdot = gr[:, 0, 1, 1]
    peri_steps = torch.cat([torch.zeros(1, dtype=torch.long), torch.nonzero((rdot[:-1] < 0) & (rdot[1:] >= 0))[:, 0]])

    r0 = params["r_infall"]
    tau_p, plunge = solve.infall(tensors.radial_infall(r0), M, 0.05)
    tau_h = solve.proper_infall_time(r0, 2 * M, M)

    drift = (equation.norm(gr, M) + 1).abs().max()
    print(f"orbit p = {p:g} M, e = {e:g}:  E = {E:.5f}, L = {L:.4f} M;  |g u u + 1| ≤ {drift:.1e} along the orbit")
    print(f"perihelion advance {math.degrees(precession):.2f}°/orbit;  weak-field 6πM/p gives "
          f"{math.degrees(equation.weak_field_precession(p, M)):.2f}°")  # fmt: skip
    print(f"ISCO at r = {equation.isco_radius(M):g} M, photon sphere at r = {equation.photon_sphere_radius(M):g} M")
    print(f"plunge from r = {r0:g} M: reaches r = {float(plunge[-1, 0, 0, 1]):.2f} M after τ = {float(tau_p[-1]):.2f} "
          f"but t = {float(plunge[-1, 0, 0, 0]):.1f};  τ to the horizon (exact) = {tau_h:.2f} GM/c³")  # fmt: skip

    orbit = {
        "gr": gr,
        "newton": newton,
        "E": E,
        "L": L,
        "precession": precession,
        "weak_deg": math.degrees(equation.weak_field_precession(p, M)),
        "peri_steps": peri_steps.numpy(),
    }
    return plot.animate(orbit, {"tau": tau_p, "traj": plunge, "tau_horizon": tau_h}, M, out_dir / "black_hole.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/black_hole_geodesics/equation.py
r"""Physics → tensor translation.

General relativity: a freely falling particle follows a geodesic of spacetime,

    \frac{d^2 x^\mu}{d\tau^2} = -\Gamma^\mu{}_{\alpha\beta}\,\dot x^\alpha \dot x^\beta      (α, β summed)

where τ is the particle's proper time and the Christoffel symbols come from the metric g_{μν}(x):

    \Gamma^\mu{}_{\alpha\beta} = ½\, g^{\mu\sigma}\left(\partial_\alpha g_{\sigma\beta}
                                 + \partial_\beta g_{\sigma\alpha} - \partial_\sigma g_{\alpha\beta}\right)   σ summed

Outside a non-rotating black hole of mass M (units G = c = 1, coordinates x^μ = (t, r, θ, φ), signature −+++)
the metric is Schwarzschild's:

    g_{\mu\nu} = \operatorname{diag}\left(-(1 - 2M/r),\; (1 - 2M/r)^{-1},\; r^2,\; r^2\sin^2\theta\right)

The horizon is at r_s = 2M. We let torch.func.jacrev differentiate g_{μν} (no hand-derived Christoffels) and
contract with einsum. For a massive particle the 4-velocity stays normalised, g_{μν} \dot x^μ \dot x^ν = −1.

Landmarks of the effective potential (per unit mass, L = angular momentum)

    ½\dot r^2 + V(r) = ½(E^2 - 1),     V = -\frac{M}{r} + \frac{L^2}{2r^2} - \frac{M L^2}{r^3}

(the last term is what Newton lacks): the innermost stable circular orbit at r = 6M, the photon sphere at
r = 3M, and a bound orbit's perihelion advancing by ≈ 6πM / (a(1 − e²)) per turn.
"""

import torch
from torch.func import jacrev, vmap


def metric(x: torch.Tensor, M: float) -> torch.Tensor:
    r"""g_{μν}(x) for one event x = (t, r, θ, φ): (4,) → (4, 4)."""
    r, theta = x[1], x[2]
    f = 1 - 2 * M / r
    return torch.diag(torch.stack([-f, 1 / f, r**2, (r * torch.sin(theta)) ** 2]))


def christoffel(x: torch.Tensor, M: float) -> torch.Tensor:
    r"""Γ^μ_{αβ} at a batch of events by automatic differentiation.   x: (B, 4) → (B, 4, 4, 4)

    dg[z, σ, β, α] = ∂_α g_{σβ}  (z = batch index; jacrev appends the derivative index last)
    """
    g = vmap(lambda p: metric(p, M))(x)  # (B, 4, 4)
    g_inv = torch.linalg.inv(g)  # (B, 4, 4)  g^{μσ}
    dg = vmap(jacrev(lambda p: metric(p, M)))(x)  # (B, 4, 4, 4)
    d1 = dg.permute(0, 1, 3, 2)  # [z, σ, α, β] = ∂_α g_{σβ}
    d2 = dg  # [z, σ, α, β] = ∂_β g_{σα}
    d3 = dg.permute(0, 3, 1, 2)  # [z, σ, α, β] = ∂_σ g_{αβ}
    return 0.5 * torch.einsum("zms,zsab->zmab", g_inv, d1 + d2 - d3)


def geodesic_acceleration(x: torch.Tensor, u: torch.Tensor, M: float) -> torch.Tensor:
    r"""\ddot x^μ = -Γ^μ_{αβ} \dot x^α \dot x^β (α, β summed).   x, u: (B, 4) → (B, 4)"""
    return -torch.einsum("zmab,za,zb->zm", christoffel(x, M), u, u)  # z = batch index


def rhs(y: torch.Tensor, M: float) -> torch.Tensor:
    """dy/dτ for y of shape (B, 2, 4): y[:, 0] = x^μ, y[:, 1] = dx^μ/dτ."""
    return torch.stack([y[:, 1], geodesic_acceleration(y[:, 0], y[:, 1], M)], dim=1)


def norm(y: torch.Tensor, M: float) -> torch.Tensor:
    r"""g_{μν} \dot x^μ \dot x^ν (= −1 for a massive particle).   y: (..., 2, 4) → (...)"""
    flat = y.reshape(-1, 2, 4)
    g = vmap(lambda p: metric(p, M))(flat[:, 0])
    return torch.einsum("zmn,zm,zn->z", g, flat[:, 1], flat[:, 1]).reshape(y.shape[:-2])


def newton_acceleration(x: torch.Tensor, M: float) -> torch.Tensor:
    r"""For comparison, Newton: \ddot x_{bi} = -M (x_{bj} x_{bj})^{-3/2} x_{bi}.   (B, 2) → (B, 2)"""
    r2 = torch.einsum("bj,bj->b", x, x)
    return -M * torch.einsum("b,bi->bi", r2**-1.5, x)


def newton_rhs(y: torch.Tensor, M: float) -> torch.Tensor:
    """dy/dt for a Newtonian orbit, y of shape (B, 2, 2) = (position, velocity) in the orbital plane."""
    return torch.stack([y[:, 1], newton_acceleration(y[:, 0], M)], dim=1)


def effective_potential(r: torch.Tensor, L: float, M: float, relativistic: bool = True) -> torch.Tensor:
    """V(r) per unit mass; relativistic=False drops the −ML²/r³ term (Newton)."""
    V = -M / r + L**2 / (2 * r**2)
    return V - M * L**2 / r**3 if relativistic else V


def isco_radius(M: float) -> float:
    """Innermost stable circular orbit, 6M (= 3 r_s)."""
    return 6 * M


def photon_sphere_radius(M: float) -> float:
    """Circular orbit of light, 3M."""
    return 3 * M


def weak_field_precession(p: float, M: float) -> float:
    """Perihelion advance per orbit 6πM/p, with p = a(1 − e²) the semi-latus rectum."""
    return 6 * torch.pi * M / p


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/black_hole_geodesics/tensors.py
"""Build the concrete tensors: initial 4-positions and 4-velocities for a bound orbit and a radial infall.

Units: G = c = 1 and M = 1, so lengths are in GM/c² (1.5 km for one solar mass) and the horizon is at r = 2.
Orbits lie in the equatorial plane θ = π/2.
"""

import math

import torch

M = 1.0


def orbit_constants(p: float, e: float) -> tuple[float, float]:
    """Energy E and angular momentum L per unit mass of the Schwarzschild orbit with semi-latus rectum p and
    eccentricity e (turning points r = p/(1 ± e)). Needs p > 6 + 2e for a stable bound orbit."""
    L = math.sqrt(p**2 * M / (p - 3 * M - e**2 * M))
    E = math.sqrt(((p - 2 * M) ** 2 - 4 * e**2 * M**2) / (p * (p - 3 * M - e**2 * M)))
    return E, L


def bound_orbit(p: float, e: float) -> torch.Tensor:
    """y0 of shape (1, 2, 4): at periapsis r = p/(1+e), moving purely in φ.

    u^μ = (E / (1 − 2M/r), 0, 0, L / r²) — the conserved E and L fix the 4-velocity.
    """
    E, L = orbit_constants(p, e)
    r = p / (1 + e)
    x = torch.tensor([0.0, r, math.pi / 2, 0.0], dtype=torch.float64)
    u = torch.tensor([E / (1 - 2 * M / r), 0.0, 0.0, L / r**2], dtype=torch.float64)
    return torch.stack([x, u])[None]


def radial_infall(r0: float) -> torch.Tensor:
    """y0 of shape (1, 2, 4): released from rest at r0, so u^μ = (1/\\sqrt{1 − 2M/r0}, 0, 0, 0)."""
    x = torch.tensor([0.0, r0, math.pi / 2, 0.0], dtype=torch.float64)
    u = torch.tensor([1 / math.sqrt(1 - 2 * M / r0), 0.0, 0.0, 0.0], dtype=torch.float64)
    return torch.stack([x, u])[None]


def newton_orbit(p: float, e: float) -> torch.Tensor:
    """y0 of shape (1, 2, 2): the Newtonian ellipse with the same p and e, at periapsis (vis-viva speed)."""
    r = p / (1 + e)
    v = math.sqrt(M * (1 + e) / r)
    return torch.tensor([[[r, 0.0], [0.0, v]]], dtype=torch.float64)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/black_hole_geodesics/solve.py
"""Integrate the geodesic equation with RK4 in proper time τ; measure perihelion advance and infall times."""

import math

import torch

from problems._common.integrators import integrate

from .equation import newton_rhs, rhs


def geodesic(y0: torch.Tensor, M: float, tau_max: float, dt: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (τ (T,), trajectory (T, B, 2, 4))."""
    n_steps = int(round(tau_max / dt))
    traj = integrate(lambda y, tau: rhs(y, M), y0, dt, n_steps)
    return torch.arange(n_steps + 1, dtype=y0.dtype) * dt, traj


def infall(y0: torch.Tensor, M: float, dt: float, r_stop: float = 2.1) -> tuple[torch.Tensor, torch.Tensor]:
    """Radial plunge from rest until just outside the horizon (r_stop in units of M).

    Schwarzschild coordinates break down at r = 2M (dt/dτ → ∞), so we stop just before it. Returns (τ, trajectory).
    """
    r0 = float(y0[0, 0, 1])
    return geodesic(y0, M, proper_infall_time(r0, r_stop * M, M), dt)


def newton(y0: torch.Tensor, M: float, t_max: float, dt: float) -> torch.Tensor:
    """The Newtonian comparison orbit, (T, B, 2, 2)."""
    return integrate(lambda y, t: newton_rhs(y, M), y0, dt, int(round(t_max / dt)))


def proper_infall_time(r0: float, r: float, M: float) -> float:
    """τ to fall from rest at r0 to r (same cycloid as Newton's radial fall): finite, even to the horizon."""
    eta = math.acos(2 * r / r0 - 1)
    return math.sqrt(r0**3 / (8 * M)) * (eta + math.sin(eta))


def periapsis_angles(traj: torch.Tensor) -> torch.Tensor:
    """φ at each periapsis passage (ṙ changes sign from − to +), linearly interpolated: (K,)."""
    rdot, phi = traj[:, 0, 1, 1], traj[:, 0, 0, 3]
    k = torch.nonzero((rdot[:-1] < 0) & (rdot[1:] >= 0)).flatten()
    f = -rdot[k] / (rdot[k + 1] - rdot[k])
    angles = phi[k] + f * (phi[k + 1] - phi[k])
    return torch.cat([phi[:1], angles])  # the run starts at periapsis


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/black_hole_geodesics/plot.py
"""Animate the precessing orbit (vs Newton), the effective potential, and a radial plunge in τ and in t."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import effective_potential, isco_radius, photon_sphere_radius


def animate(orbit: dict, plunge: dict, M: float, path: Path) -> Path:
    xyz = orbit["gr"][:, 0, 0].cpu().numpy()  # (T, 4) = (t, r, θ, φ)
    r, phi = xyz[:, 1], xyz[:, 3]
    gr = np.stack([r * np.cos(phi), r * np.sin(phi)], axis=-1)  # (T, 2)
    nw = orbit["newton"][:, 0, 0].cpu().numpy()  # (Tn, 2)
    E, L = orbit["E"], orbit["L"]
    tau_p = plunge["tau"].cpu().numpy()
    t_p, r_p = plunge["traj"][:, 0, 0, 0].cpu().numpy(), plunge["traj"][:, 0, 0, 1].cpu().numpy()

    fig = plt.figure(figsize=(8, 4.5))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.15, 1])
    ax = fig.add_subplot(gs[:, 0])
    ax_v = fig.add_subplot(gs[0, 1])
    ax_p = fig.add_subplot(gs[1, 1])

    # --- orbit in the equatorial plane ---
    reach = 1.1 * r.max()
    ax.set_xlim(-reach, reach)
    ax.set_ylim(-reach, reach)
    ax.set_aspect("equal", adjustable="box")
    ax.add_patch(plt.Circle((0, 0), 2 * M, color="black", zorder=4))
    ax.add_patch(plt.Circle((0, 0), photon_sphere_radius(M), fill=False, ls=":", color="#f59e0b", lw=1.2))
    ax.add_patch(plt.Circle((0, 0), isco_radius(M), fill=False, ls="--", color="#9ca3af", lw=1))
    ax.text(isco_radius(M) * 0.72, isco_radius(M) * 0.72, "ISCO", fontsize=7, color="#6b7280")
    ax.plot(nw[:, 0], nw[:, 1], color="#9ca3af", lw=1, label="Newton (closed ellipse)")
    trail = ax.plot([], [], color=ACCENT[0], lw=1.3, label="geodesic (GR)")[0]
    dot = ax.plot([], [], "o", color=ACCENT[0], ms=5)[0]
    peri = ax.plot([], [], "x", color=ACCENT[1], ms=8, mew=2, label="periapsis")[0]
    ax.legend(fontsize=7, loc="upper right")
    ax.set_xlabel("x  (GM/c²)")
    ax.set_ylabel("y  (GM/c²)")
    dphi = orbit["precession"]
    ax.set_title(f"perihelion advance {np.degrees(dphi):.0f}°/orbit (weak field {orbit['weak_deg']:.0f}°)", fontsize=9)

    # --- effective potential ---
    rr = torch.linspace(2.2 * M, 1.3 * r.max(), 400, dtype=torch.float64)
    V_gr, V_newton = effective_potential(rr, L, M).numpy(), effective_potential(rr, L, M, relativistic=False).numpy()
    V_isco = effective_potential(rr, (12**0.5) * M, M).numpy()
    rr = rr.numpy()
    ax_v.plot(rr, V_gr, color=ACCENT[0], lw=1.5, label="GR")
    ax_v.plot(rr, V_newton, color="#9ca3af", lw=1, label="Newton")
    ax_v.plot(rr, V_isco, color=ACCENT[3], lw=1, ls="--", label="L = √12 M (ISCO)")
    level = 0.5 * (E**2 - 1)
    ax_v.axhline(level, color=ACCENT[1], lw=1)
    ax_v.axvline(isco_radius(M), color="#9ca3af", ls="--", lw=0.8)
    ax_v.axvline(photon_sphere_radius(M), color="#f59e0b", ls=":", lw=1)
    ax_v.set_ylim(level - 0.06, level + 0.08)
    ax_v.set_xlim(rr[0], rr[-1])
    ax_v.set_ylabel("V(r)")
    ax_v.legend(fontsize=6, loc="lower right")
    marker = ax_v.plot([], [], "o", color=ACCENT[0], ms=5)[0]

    # --- radial plunge: proper time vs coordinate time ---
    ax_p.axhline(2 * M, color="black", lw=1.2)
    ax_p.text(t_p[-1] * 0.98, 2.2 * M, "horizon r = 2M", ha="right", fontsize=7)
    ax_p.set_xlim(0, t_p[-1] * 1.05)
    ax_p.set_ylim(0, r_p[0] * 1.1)
    ax_p.set_xlabel("time  (GM/c³)")
    ax_p.set_ylabel("r")
    ax_p.axvline(plunge["tau_horizon"], color=ACCENT[2], ls=":", lw=1)
    run_tau = ax_p.plot([], [], color=ACCENT[2], lw=1.8, label="r(τ) proper time: finite")[0]
    run_t = ax_p.plot([], [], color=ACCENT[4], lw=1.8, label="r(t) far observer: freezes")[0]
    ax_p.legend(fontsize=6, loc="upper right")

    n_orbit, n_plunge = len(r), len(tau_p)
    frames = frame_indices(n_orbit, 100)
    peri_xy = gr[orbit["peri_steps"]]

    def update(step: int):
        trail.set_data(gr[: step + 1, 0], gr[: step + 1, 1])
        dot.set_data([gr[step, 0]], [gr[step, 1]])
        shown = orbit["peri_steps"] <= step
        peri.set_data(peri_xy[shown, 0], peri_xy[shown, 1])
        marker.set_data([r[step]], [level])
        k = min(n_plunge - 1, round(step / max(1, n_orbit - 1) * 1.6 * (n_plunge - 1)))  # plunge finishes early
        run_tau.set_data(tau_p[: k + 1], r_p[: k + 1])
        run_t.set_data(t_p[: k + 1], r_p[: k + 1])

    fig.tight_layout()
    return save_gif(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'p': 14.0,
    'e': 0.4,
    'n_orbits': 3.0,
    'dt': 2.0,
    'r_infall': 10.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.black_hole_geodesics")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.black_hole_geodesics.main")
problem_dir = Path("problems/astro/black_hole_geodesics")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
